In [6]:
!pip install torch torchvision opencv-python albumentations matplotlib scikit-learn pillow pydicom pynrrd

Defaulting to user installation because normal site-packages is not writeable
Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com

[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: python3 -m pip install --upgrade pip


In [3]:
# Run this to understand your data better
import json
from pathlib import Path
import nrrd
import numpy as np

for patient_dir in list(Path('data/train').iterdir())[:3]:
    print(f"\n{'='*60}")
    print(f"Patient: {patient_dir.name}")
    print('='*60)
    
    # Check JSON
    with open(patient_dir / 'patient_data.json') as f:
        data = json.load(f)
    
    print(f"\nClinical Data:")
    print(f"  Tumor Grade: {data['demographic_clinical'].get('Tumor Grade')}")
    print(f"  Staging: T{data['demographic_clinical'].get('Staging(Tumor Size)#[T]')}")
    print(f"  ER/PR/HER2: {data['demographic_clinical'].get('ER')}/{data['demographic_clinical'].get('PR')}/{data['demographic_clinical'].get('HER2')}")
    
    print(f"\nRadiomic Features:")
    print(f"  Tumor Volume: {data['imaging_features'].get('Volume_cu_mm_Tumor')} mm³")
    print(f"  Tumor Size: {data['imaging_features'].get('TumorMajorAxisLength_mm')} mm")
    
    print(f"\nAnnotations:")
    annotations = data.get('annotations', [])
    print(f"  Number of annotations: {len(annotations)}")
    if annotations:
        print(f"  First annotation keys: {list(annotations[0].keys())}")
    
    print(f"\nMasks:")
    seg_dir = patient_dir / 'Segmentation_Masks_NRRD'
    if seg_dir.exists():
        for mask_file in seg_dir.glob('*.nrrd'):
            mask_data, _ = nrrd.read(str(mask_file))
            coverage = (mask_data > 0).sum() / mask_data.size * 100
            print(f"  {mask_file.name}: {mask_data.shape}, {coverage:.1f}% annotated")


Patient: Patient_117

Clinical Data:
  Tumor Grade: 3
  Staging: TNone
  ER/PR/HER2: 0/0/0

Radiomic Features:
  Tumor Volume: 3368.269330613 mm³
  Tumor Size: 25.3104221125885 mm

Annotations:
  Number of annotations: 1
  First annotation keys: ['patient_id', 'bounding_box_3d', 'annotation_type']

Masks:
  v2_dense_tissue_train_Breast_MRI_117_pre_108.nrrd: (512, 512, 1), 1.6% annotated
  v2_breast_train_Breast_MRI_117_pre_035.nrrd: (512, 512, 1), 3.6% annotated
  v2_breast_train_Breast_MRI_117_pre_108.nrrd: (512, 512, 1), 5.7% annotated
  v2_breast_train_Breast_MRI_117_pre_164.nrrd: (512, 512, 1), 3.1% annotated
  v2_dense_tissue_train_Breast_MRI_117_pre_147.nrrd: (512, 512, 1), 0.5% annotated
  v2_dense_tissue_train_Breast_MRI_117_pre_091.nrrd: (512, 512, 1), 1.7% annotated

Patient: Patient_876

Clinical Data:
  Tumor Grade: 3
  Staging: TNone
  ER/PR/HER2: 1/1/0

Radiomic Features:
  Tumor Volume: 186.74367264 mm³
  Tumor Size: 12.3332769087163 mm

Annotations:
  Number of annotat

In [9]:
"""
DUKE MULTI-TASK MULTIMODAL MODEL
=================================
Simultaneously performs:
1. Tumor Detection (bounding box regression)
2. Tumor Classification (grade, subtype, receptors)
3. Tumor Segmentation (pixel-wise mask)

This is the COMPLETE solution that leverages all multimodal data!
"""

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from torchvision.models import resnet50

import numpy as np
import json
from pathlib import Path
from PIL import Image, ImageDraw
import pydicom
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')


# ==================== MULTI-TASK DATASET ====================

class DukeMultiTaskDataset(Dataset):
    """
    Dataset for multi-task learning:
    - Bounding box detection
    - Tumor classification  
    - Tumor segmentation
    """
    
    def __init__(self, patients_dir, transform=None, sample_slices=5,
                 use_clinical=True, use_imaging_features=True):
        self.patients_dir = Path(patients_dir)
        self.transform = transform
        self.sample_slices = sample_slices
        self.use_clinical = use_clinical
        self.use_imaging_features = use_imaging_features
        
        self.samples = []
        self.load_dataset()
        
    def load_dataset(self):
        """Load dataset with all labels"""
        patient_folders = [d for d in self.patients_dir.iterdir() if d.is_dir()]
        
        clinical_features_list = []
        imaging_features_list = []
        
        valid_samples = 0
        with_bbox = 0
        
        for patient_dir in sorted(patient_folders):
            data_file = patient_dir / 'patient_data.json'
            
            if not data_file.exists():
                continue
            
            try:
                with open(data_file) as f:
                    metadata = json.load(f)
                
                mri_dir = patient_dir / 'MRI_DICOM_sample'
                if not mri_dir.exists():
                    continue
                    
                dicom_files = sorted(list(mri_dir.glob('**/*.dcm')))
                if not dicom_files:
                    continue
                
                # Get bounding box
                annotations = metadata.get('annotations', [])
                bbox_3d = None
                has_bbox = False
                
                if annotations and len(annotations) > 0:
                    ann = annotations[0]
                    if 'bounding_box_3d' in ann:
                        bbox_3d = ann['bounding_box_3d']
                        if isinstance(bbox_3d, dict):
                            # Validate bbox
                            x_min = bbox_3d.get('start_column', 0)
                            x_max = bbox_3d.get('end_column', 0)
                            y_min = bbox_3d.get('start_row', 0)
                            y_max = bbox_3d.get('end_row', 0)
                            if x_max > x_min and y_max > y_min:
                                has_bbox = True
                                with_bbox += 1
                
                # Extract features and labels
                clinical_features = self._extract_clinical_features(metadata)
                imaging_features = self._extract_imaging_features(metadata)
                classification_labels = self._extract_classification_labels(metadata)
                
                if classification_labels is None:
                    continue
                
                sample_info = {
                    'patient_id': patient_dir.name,
                    'dicom_files': dicom_files,
                    'clinical_features': clinical_features,
                    'imaging_features': imaging_features,
                    'bbox_3d': bbox_3d,
                    'has_bbox': has_bbox,
                    'classification_labels': classification_labels,
                    'metadata': metadata
                }
                
                self.samples.append(sample_info)
                valid_samples += 1
                
                if clinical_features is not None:
                    clinical_features_list.append(clinical_features)
                if imaging_features is not None:
                    imaging_features_list.append(imaging_features)
                
            except Exception as e:
                print(f"Error loading {patient_dir.name}: {e}")
                continue
        
        # Feature normalization
        if clinical_features_list:
            self.clinical_mean = np.mean(clinical_features_list, axis=0)
            self.clinical_std = np.std(clinical_features_list, axis=0) + 1e-8
        else:
            self.clinical_mean = np.zeros(10)
            self.clinical_std = np.ones(10)
            
        if imaging_features_list:
            self.imaging_mean = np.mean(imaging_features_list, axis=0)
            self.imaging_std = np.std(imaging_features_list, axis=0) + 1e-8
        else:
            self.imaging_mean = np.zeros(12)
            self.imaging_std = np.ones(12)
        
        print(f"\nDataset Statistics:")
        print(f"  Total samples: {valid_samples}")
        print(f"  Samples with bounding boxes: {with_bbox}")
    
    def _extract_clinical_features(self, metadata):
        """Extract clinical features (excluding labels)"""
        if not self.use_clinical:
            return None
            
        clinical = metadata.get('demographic_clinical', {})
        if not clinical:
            return None
        
        feature_keys = [
            'Age at last contact in EMR f/u(days)(from the date of diagnosis) ,last time patient known to be alive, unless age of death is reported(in such case the age of death',
            'Menopause (at diagnosis)',
            'Days to MRI (From the Date of Diagnosis)',
            'Field Strength (Tesla)',
            'TR (Repetition Time)',
            'TE (Echo Time)',
            'Slice Thickness ',
            'Staging(Nodes)#(Nx replaced by -1)[N]',
            'Staging(Metastasis)#(Mx -replaced by -1)[M]',
            'Staging(Tumor Size)#[T]'
        ]
        
        features = []
        for key in feature_keys:
            val = clinical.get(key, 0)
            if isinstance(val, (int, float)) and not np.isnan(val):
                features.append(float(val))
            else:
                features.append(0.0)
        
        return np.array(features, dtype=np.float32)
    
    def _extract_imaging_features(self, metadata):
        """Extract radiomic features"""
        if not self.use_imaging_features:
            return None
            
        imaging = metadata.get('imaging_features', {})
        if not imaging:
            return None
        
        feature_keys = [
            'TumorMajorAxisLength_mm',
            'Volume_cu_mm_Tumor',
            'Energy_Tumor',
            'Contrast_Tumor',
            'Homogeneity1_Tumor',
            'Max_Enhancement_from_char_curv',
            'Time_to_Peak_from_char_curv',
            'Uptake_rate_from_char_curv',
            'Washout_rate_from_char_curv',
            'breastDensity_T1',
            'breastDensity_PostCon',
            'Peak_SER_tumor'
        ]
        
        features = []
        for key in feature_keys:
            val = imaging.get(key, 0)
            if isinstance(val, (int, float)) and not np.isnan(val):
                features.append(float(val))
            else:
                features.append(0.0)
        
        return np.array(features, dtype=np.float32)
    
    def _extract_classification_labels(self, metadata):
        """Extract classification labels"""
        clinical = metadata.get('demographic_clinical', {})
        if not clinical:
            return None
        
        # Tumor Grade (0, 1, 2 for grades 1, 2, 3)
        grade = clinical.get('Tumor Grade', 3)
        if not isinstance(grade, (int, float)) or grade not in [1, 2, 3]:
            grade = 3
        grade = int(grade) - 1
        
        # ER/PR/HER2 status
        er = int(clinical.get('ER', 0))
        pr = int(clinical.get('PR', 0))
        her2 = int(clinical.get('HER2', 0))
        
        # Molecular Subtype
        if er == 0 and pr == 0 and her2 == 0:
            subtype = 0  # Triple Negative
        elif her2 == 1:
            subtype = 1  # HER2+
        else:
            subtype = 2  # Luminal
        
        # High grade binary
        high_grade = 1 if grade == 2 else 0
        
        return {
            'grade': grade,
            'subtype': subtype,
            'high_grade': high_grade,
            'er': er,
            'pr': pr,
            'her2': her2
        }
    
    def _create_bbox_and_mask(self, bbox_3d, img_size=224):
        """
        Create both normalized bounding box coordinates AND segmentation mask.
        Returns: bbox_norm (x_min, y_min, x_max, y_max in [0,1]), mask (HxW)
        """
        if bbox_3d is None or not isinstance(bbox_3d, dict):
            # Return empty bbox and mask
            return torch.tensor([0., 0., 0., 0.]), torch.zeros((img_size, img_size), dtype=torch.long), False
        
        try:
            # Extract coordinates (original 512x512 space)
            x_min = int(bbox_3d.get('start_column', 0))
            x_max = int(bbox_3d.get('end_column', 0))
            y_min = int(bbox_3d.get('start_row', 0))
            y_max = int(bbox_3d.get('end_row', 0))
            
            # Validate
            if x_max <= x_min or y_max <= y_min:
                return torch.tensor([0., 0., 0., 0.]), torch.zeros((img_size, img_size), dtype=torch.long), False
            
            # Normalize to [0, 1] for bbox
            original_size = 512.0
            bbox_norm = torch.tensor([
                x_min / original_size,
                y_min / original_size,
                x_max / original_size,
                y_max / original_size
            ], dtype=torch.float32)
            
            # Create segmentation mask
            mask = np.zeros((512, 512), dtype=np.uint8)
            mask[y_min:y_max, x_min:x_max] = 1
            
            # Resize mask to target size
            mask_pil = Image.fromarray(mask * 255, mode='L')
            mask_pil = mask_pil.resize((img_size, img_size), Image.NEAREST)
            mask_array = np.array(mask_pil)
            mask_array = (mask_array > 127).astype(np.int64)
            mask_tensor = torch.from_numpy(mask_array).long()
            
            has_annotation = mask_tensor.sum() > 0
            
            return bbox_norm, mask_tensor, has_annotation
            
        except Exception as e:
            print(f"Error creating bbox/mask: {e}")
            return torch.tensor([0., 0., 0., 0.]), torch.zeros((img_size, img_size), dtype=torch.long), False
    
    def _load_dicom_image(self, dicom_path):
        """Load DICOM"""
        try:
            dcm = pydicom.dcmread(str(dicom_path))
            image = dcm.pixel_array
            
            while len(image.shape) > 2 and image.shape[0] == 1:
                image = image.squeeze(0)
            
            if len(image.shape) == 3:
                if image.shape[0] < image.shape[-1]:
                    image = image[image.shape[0] // 2]
                else:
                    if image.shape[-1] in [1, 3, 4]:
                        image = image[..., 0]
                    else:
                        image = image[:, :, image.shape[-1] // 2]
            
            if len(image.shape) != 2:
                return None
            
            if image.min() == image.max():
                image = np.zeros_like(image)
            else:
                image = ((image - image.min()) / (image.max() - image.min()) * 255).astype(np.uint8)
            
            return image
        except:
            return None
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        sample = self.samples[idx]
        
        # Load MRI
        dicom_files = sample['dicom_files']
        num_files = len(dicom_files)
        
        if num_files == 0:
            image = Image.new('RGB', (224, 224), color=0)
        else:
            indices = np.linspace(0, num_files-1, min(self.sample_slices, num_files), dtype=int)
            
            images = []
            for i in indices:
                img = self._load_dicom_image(dicom_files[i])
                if img is not None and len(img.shape) == 2:
                    images.append(img)
            
            if not images:
                image = Image.new('RGB', (224, 224), color=0)
            else:
                stacked = np.stack(images, axis=0)
                avg_slice = stacked.mean(axis=0).astype(np.uint8)
                
                if len(avg_slice.shape) == 2:
                    image = Image.fromarray(avg_slice, mode='L').convert('RGB')
                else:
                    image = Image.new('RGB', (224, 224), color=0)
        
        if self.transform:
            image = self.transform(image)
        
        # Features
        clinical_feat = sample['clinical_features']
        imaging_feat = sample['imaging_features']
        
        if clinical_feat is not None and self.use_clinical:
            clinical_feat = (clinical_feat - self.clinical_mean) / self.clinical_std
            clinical_feat = torch.FloatTensor(clinical_feat)
        else:
            clinical_feat = torch.zeros(10)
        
        if imaging_feat is not None and self.use_imaging_features:
            imaging_feat = (imaging_feat - self.imaging_mean) / self.imaging_std
            imaging_feat = torch.FloatTensor(imaging_feat)
        else:
            imaging_feat = torch.zeros(12)
        
        # Bounding box and mask
        bbox_norm, mask, has_bbox = self._create_bbox_and_mask(sample['bbox_3d'])
        
        # Classification labels
        labels = sample['classification_labels']
        
        return {
            'image': image,
            'clinical': clinical_feat,
            'imaging': imaging_feat,
            'bbox': bbox_norm,
            'mask': mask,
            'has_bbox': torch.tensor(1.0 if has_bbox else 0.0),
            'grade': labels['grade'],
            'subtype': labels['subtype'],
            'high_grade': labels['high_grade'],
            'patient_id': sample['patient_id']
        }


# ==================== MULTI-TASK MODEL ====================

class MultiTaskMultimodalModel(nn.Module):
    """
    Multi-task model that simultaneously:
    1. Detects tumor (bounding box)
    2. Classifies tumor (grade, subtype)
    3. Segments tumor (pixel-wise mask)
    
    Uses shared encoder with task-specific heads.
    """
    
    def __init__(self, clinical_dim=10, imaging_dim=12):
        super().__init__()
        
        # Shared image encoder (ResNet50)
        resnet = resnet50(pretrained=True)
        self.conv1 = resnet.conv1
        self.bn1 = resnet.bn1
        self.relu = resnet.relu
        self.maxpool = resnet.maxpool
        self.layer1 = resnet.layer1  # 256 channels
        self.layer2 = resnet.layer2  # 512 channels
        self.layer3 = resnet.layer3  # 1024 channels
        self.layer4 = resnet.layer4  # 2048 channels
        
        # Tabular encoder
        tabular_dim = clinical_dim + imaging_dim
        self.tabular_encoder = nn.Sequential(
            nn.Linear(tabular_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 512)
        )
        
        # Global pooling for classification/detection tasks
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        
        # Fusion for classification/detection
        fusion_dim = 2048 + 512
        self.fusion = nn.Sequential(
            nn.Linear(fusion_dim, 1024),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Dropout(0.3)
        )
        
        # Task 1: Bounding Box Detection (x_min, y_min, x_max, y_max)
        self.bbox_head = nn.Sequential(
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, 4),
            nn.Sigmoid()  # Normalize to [0, 1]
        )
        
        # Task 2: Classification heads
        self.grade_head = nn.Linear(512, 3)  # 3 grades
        self.subtype_head = nn.Linear(512, 3)  # 3 subtypes
        self.high_grade_head = nn.Linear(512, 2)  # binary
        
        # Task 3: Segmentation decoder (U-Net style)
        self.up1 = nn.ConvTranspose2d(2048, 1024, 2, stride=2)
        self.dec1 = self._decoder_block(1024 + 1024, 512)
        self.up2 = nn.ConvTranspose2d(512, 512, 2, stride=2)
        self.dec2 = self._decoder_block(512 + 512, 256)
        self.up3 = nn.ConvTranspose2d(256, 256, 2, stride=2)
        self.dec3 = self._decoder_block(256 + 256, 128)
        self.up4 = nn.ConvTranspose2d(128, 128, 2, stride=2)
        self.dec4 = self._decoder_block(128 + 64, 64)
        self.up5 = nn.ConvTranspose2d(64, 64, 2, stride=2)
        self.seg_out = nn.Conv2d(64, 2, 1)  # 2 classes: background, tumor
    
    def _decoder_block(self, in_ch, out_ch):
        return nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True)
        )
    
    def forward(self, image, clinical, imaging):
        # Encode image with skip connections for segmentation
        x = self.conv1(image)
        x = self.bn1(x)
        x = self.relu(x)
        x0 = x  # 64 channels, 112x112
        
        x = self.maxpool(x)
        x1 = self.layer1(x)  # 256 channels, 56x56
        x2 = self.layer2(x1)  # 512 channels, 28x28
        x3 = self.layer3(x2)  # 1024 channels, 14x14
        x4 = self.layer4(x3)  # 2048 channels, 7x7
        
        # Encode tabular features
        tabular = torch.cat([clinical, imaging], dim=1)
        tab_feat = self.tabular_encoder(tabular)
        
        # Classification/Detection branch
        img_global = self.avgpool(x4).view(x4.size(0), -1)
        fused = torch.cat([img_global, tab_feat], dim=1)
        fused = self.fusion(fused)
        
        # Task outputs
        bbox = self.bbox_head(fused)
        grade = self.grade_head(fused)
        subtype = self.subtype_head(fused)
        high_grade = self.high_grade_head(fused)
        
        # Segmentation branch (decoder with skip connections)
        d1 = self.up1(x4)
        d1 = torch.cat([d1, x3], dim=1)
        d1 = self.dec1(d1)
        
        d2 = self.up2(d1)
        d2 = torch.cat([d2, x2], dim=1)
        d2 = self.dec2(d2)
        
        d3 = self.up3(d2)
        d3 = torch.cat([d3, x1], dim=1)
        d3 = self.dec3(d3)
        
        d4 = self.up4(d3)
        d4 = torch.cat([d4, x0], dim=1)
        d4 = self.dec4(d4)
        
        d5 = self.up5(d4)
        seg = self.seg_out(d5)
        
        return {
            'bbox': bbox,
            'grade': grade,
            'subtype': subtype,
            'high_grade': high_grade,
            'segmentation': seg
        }


# ==================== TRAINER ====================

class MultiTaskTrainer:
    """Trainer for multi-task learning"""
    
    def __init__(self, model, device, lr=1e-4):
        self.model = model
        self.device = device
        
        # Loss functions
        self.bbox_criterion = nn.SmoothL1Loss()
        self.class_criterion = nn.CrossEntropyLoss()
        self.seg_criterion = nn.CrossEntropyLoss()
        
        self.optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
        self.scheduler = optim.lr_scheduler.ReduceLROnPlateau(
            self.optimizer, mode='min', factor=0.5, patience=3, verbose=True
        )
        
        self.history = {
            'train_loss': [], 'val_loss': [],
            'bbox_loss': [], 'class_loss': [], 'seg_loss': [],
            'grade_acc': [], 'seg_dice': []
        }
        self.best_loss = float('inf')
    
    def train_epoch(self, train_loader):
        self.model.train()
        total_loss = 0.0
        
        for batch in train_loader:
            # Move to device
            images = batch['image'].to(self.device)
            clinical = batch['clinical'].to(self.device)
            imaging = batch['imaging'].to(self.device)
            bbox_true = batch['bbox'].to(self.device)
            mask_true = batch['mask'].to(self.device)
            has_bbox = batch['has_bbox'].to(self.device)
            grade = torch.tensor(batch['grade']).to(self.device)
            subtype = torch.tensor(batch['subtype']).to(self.device)
            high_grade = torch.tensor(batch['high_grade']).to(self.device)
            
            self.optimizer.zero_grad()
            
            # Forward
            outputs = self.model(images, clinical, imaging)
            
            # Compute losses
            bbox_loss = self.bbox_criterion(outputs['bbox'], bbox_true) * has_bbox.mean()
            grade_loss = self.class_criterion(outputs['grade'], grade)
            subtype_loss = self.class_criterion(outputs['subtype'], subtype)
            high_grade_loss = self.class_criterion(outputs['high_grade'], high_grade)
            seg_loss = self.seg_criterion(outputs['segmentation'], mask_true) * has_bbox.mean()
            
            # Combined loss with weights
            loss = (0.3 * bbox_loss + 
                    0.2 * grade_loss + 
                    0.2 * subtype_loss + 
                    0.1 * high_grade_loss +
                    0.2 * seg_loss)
            
            loss.backward()
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
            self.optimizer.step()
            
            total_loss += loss.item()
        
        return total_loss / len(train_loader)
    
    def validate(self, val_loader):
        self.model.eval()
        total_loss = 0.0
        bbox_losses = []
        class_losses = []
        seg_losses = []
        grade_correct = 0
        total = 0
        dice_scores = []
        
        with torch.no_grad():
            for batch in val_loader:
                images = batch['image'].to(self.device)
                clinical = batch['clinical'].to(self.device)
                imaging = batch['imaging'].to(self.device)
                bbox_true = batch['bbox'].to(self.device)
                mask_true = batch['mask'].to(self.device)
                has_bbox = batch['has_bbox'].to(self.device)
                grade = torch.tensor(batch['grade']).to(self.device)
                subtype = torch.tensor(batch['subtype']).to(self.device)
                high_grade = torch.tensor(batch['high_grade']).to(self.device)
                
                outputs = self.model(images, clinical, imaging)
                
                bbox_loss = self.bbox_criterion(outputs['bbox'], bbox_true) * has_bbox.mean()
                grade_loss = self.class_criterion(outputs['grade'], grade)
                subtype_loss = self.class_criterion(outputs['subtype'], subtype)
                high_grade_loss = self.class_criterion(outputs['high_grade'], high_grade)
                seg_loss = self.seg_criterion(outputs['segmentation'], mask_true) * has_bbox.mean()
                
                loss = (0.3 * bbox_loss + 0.2 * grade_loss + 0.2 * subtype_loss + 
                        0.1 * high_grade_loss + 0.2 * seg_loss)
                
                total_loss += loss.item()
                bbox_losses.append(bbox_loss.item())
                class_losses.append((grade_loss + subtype_loss + high_grade_loss).item() / 3)
                seg_losses.append(seg_loss.item())
                
                # Grade accuracy
                grade_pred = torch.argmax(outputs['grade'], dim=1)
                grade_correct += (grade_pred == grade).sum().item()
                total += len(grade)
                
                # Segmentation Dice
                seg_pred = torch.argmax(outputs['segmentation'], dim=1)
                for i in range(len(mask_true)):
                    if has_bbox[i] > 0:
                        pred = (seg_pred[i] == 1).float()
                        target = (mask_true[i] == 1).float()
                        intersection = (pred * target).sum()
                        dice = (2. * intersection + 1e-6) / (pred.sum() + target.sum() + 1e-6)
                        dice_scores.append(dice.item())
        
        val_loss = total_loss / len(val_loader)
        grade_acc = grade_correct / total
        avg_dice = np.mean(dice_scores) if dice_scores else 0.0
        
        return (val_loss, np.mean(bbox_losses), np.mean(class_losses), 
                np.mean(seg_losses), grade_acc, avg_dice)
    
    def train(self, train_loader, val_loader, num_epochs=50, patience=10):
        print("\n" + "="*70)
        print("MULTI-TASK TRAINING")
        print("="*70)
        print("Tasks: Detection + Classification + Segmentation")
        print("="*70)
        
        patience_counter = 0
        
        for epoch in range(num_epochs):
            train_loss = self.train_epoch(train_loader)
            val_loss, bbox_loss, class_loss, seg_loss, grade_acc, seg_dice = self.validate(val_loader)
            
            self.history['train_loss'].append(train_loss)
            self.history['val_loss'].append(val_loss)
            self.history['bbox_loss'].append(bbox_loss)
            self.history['class_loss'].append(class_loss)
            self.history['seg_loss'].append(seg_loss)
            self.history['grade_acc'].append(grade_acc)
            self.history['seg_dice'].append(seg_dice)
            
            self.scheduler.step(val_loss)
            
            print(f"\nEpoch {epoch+1}/{num_epochs}")
            print(f"  Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")
            print(f"  BBox Loss:  {bbox_loss:.4f}")
            print(f"  Class Loss: {class_loss:.4f}")
            print(f"  Seg Loss:   {seg_loss:.4f}")
            print(f"  Grade Acc:  {grade_acc:.4f}")
            print(f"  Seg Dice:   {seg_dice:.4f}")
            
            if val_loss < self.best_loss:
                self.best_loss = val_loss
                torch.save({
                    'epoch': epoch,
                    'model_state_dict': self.model.state_dict(),
                    'best_loss': self.best_loss,
                }, 'best_multitask_model.pth')
                patience_counter = 0
                print(f"  ✓ Best model saved!")
            else:
                patience_counter += 1
                print(f"  Patience: {patience_counter}/{patience}")
            
            if patience_counter >= patience:
                print(f"\nEarly stopping at epoch {epoch+1}")
                break
        
        if self.best_loss < float('inf'):
            checkpoint = torch.load('best_multitask_model.pth')
            self.model.load_state_dict(checkpoint['model_state_dict'])
            print(f"\nTraining complete! Best Loss: {self.best_loss:.4f}")


# ==================== VISUALIZATION ====================

def visualize_multitask_results(model, test_loader, device, num_samples=4):
    """Visualize all three tasks together"""
    model.eval()
    
    fig = plt.figure(figsize=(20, 5*num_samples))
    
    sample_count = 0
    with torch.no_grad():
        for batch in test_loader:
            if sample_count >= num_samples:
                break
            
            images = batch['image'].to(device)
            clinical = batch['clinical'].to(device)
            imaging = batch['imaging'].to(device)
            bbox_true = batch['bbox']
            mask_true = batch['mask']
            grade_true = batch['grade']
            
            outputs = model(images, clinical, imaging)
            
            for i in range(len(images)):
                if sample_count >= num_samples:
                    break
                
                # Prepare image
                img = images[i].cpu().permute(1, 2, 0).numpy()
                img = (img - img.min()) / (img.max() - img.min() + 1e-8)
                
                # Get predictions
                bbox_pred = outputs['bbox'][i].cpu().numpy()
                bbox_gt = bbox_true[i].numpy()
                
                seg_pred = torch.argmax(outputs['segmentation'][i], dim=0).cpu().numpy()
                seg_gt = mask_true[i].numpy()
                
                grade_pred = torch.argmax(outputs['grade'][i]).item()
                grade_gt = grade_true[i]
                
                subtype_pred = torch.argmax(outputs['subtype'][i]).item()
                high_grade_pred = torch.argmax(outputs['high_grade'][i]).item()
                
                # Create subplot grid: 5 columns
                base_idx = sample_count * 5 + 1
                
                # Column 1: Original image
                ax1 = plt.subplot(num_samples, 5, base_idx)
                ax1.imshow(img)
                ax1.set_title(f'MRI\n{batch["patient_id"][i]}')
                ax1.axis('off')
                
                # Column 2: Bounding box detection
                ax2 = plt.subplot(num_samples, 5, base_idx + 1)
                ax2.imshow(img)
                
                # Draw ground truth bbox (green)
                if bbox_gt.sum() > 0:
                    x_min_gt, y_min_gt, x_max_gt, y_max_gt = bbox_gt * 224
                    w_gt = x_max_gt - x_min_gt
                    h_gt = y_max_gt - y_min_gt
                    rect_gt = patches.Rectangle((x_min_gt, y_min_gt), w_gt, h_gt,
                                                linewidth=2, edgecolor='green', 
                                                facecolor='none', label='GT')
                    ax2.add_patch(rect_gt)
                
                # Draw predicted bbox (red)
                x_min_pred, y_min_pred, x_max_pred, y_max_pred = bbox_pred * 224
                w_pred = x_max_pred - x_min_pred
                h_pred = y_max_pred - y_min_pred
                rect_pred = patches.Rectangle((x_min_pred, y_min_pred), w_pred, h_pred,
                                            linewidth=2, edgecolor='red', 
                                            facecolor='none', linestyle='--', label='Pred')
                ax2.add_patch(rect_pred)
                
                ax2.set_title('Detection\n(Green=GT, Red=Pred)')
                ax2.axis('off')
                ax2.legend(loc='upper right', fontsize=8)
                
                # Column 3: Segmentation GT
                ax3 = plt.subplot(num_samples, 5, base_idx + 2)
                ax3.imshow(img, cmap='gray')
                ax3.imshow(seg_gt, alpha=0.5, cmap='Reds')
                ax3.set_title('Segmentation GT')
                ax3.axis('off')
                
                # Column 4: Segmentation Prediction
                ax4 = plt.subplot(num_samples, 5, base_idx + 3)
                ax4.imshow(img, cmap='gray')
                ax4.imshow(seg_pred, alpha=0.5, cmap='Greens')
                ax4.set_title('Segmentation Pred')
                ax4.axis('off')
                
                # Column 5: Classification results
                ax5 = plt.subplot(num_samples, 5, base_idx + 4)
                ax5.axis('off')
                
                # Classification text
                grade_names = ['Grade 1', 'Grade 2', 'Grade 3']
                subtype_names = ['Triple Neg', 'HER2+', 'Luminal']
                
                info_text = f"CLASSIFICATION\n\n"
                info_text += f"Grade:\n"
                info_text += f"  GT: {grade_names[grade_gt]}\n"
                info_text += f"  Pred: {grade_names[grade_pred]}\n"
                info_text += f"  {'✓' if grade_pred == grade_gt else '✗'}\n\n"
                info_text += f"Subtype:\n"
                info_text += f"  Pred: {subtype_names[subtype_pred]}\n\n"
                info_text += f"High Grade:\n"
                info_text += f"  Pred: {'Yes' if high_grade_pred == 1 else 'No'}"
                
                ax5.text(0.1, 0.5, info_text, fontsize=10, 
                        verticalalignment='center', family='monospace',
                        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
                ax5.set_title('Classification')
                
                sample_count += 1
    
    plt.tight_layout()
    plt.savefig('multitask_results.png', dpi=300, bbox_inches='tight')
    print("\n✓ Multi-task results saved to multitask_results.png")
    plt.close()


def plot_training_curves(history):
    """Plot training history"""
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    
    # Overall loss
    axes[0, 0].plot(history['train_loss'], label='Train', marker='o')
    axes[0, 0].plot(history['val_loss'], label='Val', marker='s')
    axes[0, 0].set_title('Overall Loss')
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('Loss')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    
    # Task-specific losses
    axes[0, 1].plot(history['bbox_loss'], label='BBox', marker='o', color='blue')
    axes[0, 1].plot(history['class_loss'], label='Class', marker='s', color='orange')
    axes[0, 1].plot(history['seg_loss'], label='Seg', marker='^', color='green')
    axes[0, 1].set_title('Task-Specific Losses')
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].set_ylabel('Loss')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)
    
    # Grade accuracy
    axes[0, 2].plot(history['grade_acc'], label='Grade Acc', marker='o', color='purple')
    axes[0, 2].set_title('Classification Accuracy')
    axes[0, 2].set_xlabel('Epoch')
    axes[0, 2].set_ylabel('Accuracy')
    axes[0, 2].legend()
    axes[0, 2].grid(True, alpha=0.3)
    
    # Segmentation Dice
    axes[1, 0].plot(history['seg_dice'], label='Dice Score', marker='o', color='green')
    axes[1, 0].set_title('Segmentation Dice Score')
    axes[1, 0].set_xlabel('Epoch')
    axes[1, 0].set_ylabel('Dice')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)
    
    # All metrics together
    axes[1, 1].plot(history['grade_acc'], label='Grade Acc', marker='o')
    axes[1, 1].plot(history['seg_dice'], label='Seg Dice', marker='s')
    axes[1, 1].set_title('All Metrics')
    axes[1, 1].set_xlabel('Epoch')
    axes[1, 1].set_ylabel('Score')
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)
    
    # Summary statistics
    axes[1, 2].axis('off')
    final_stats = f"""
    FINAL PERFORMANCE
    
    Grade Accuracy: {history['grade_acc'][-1]:.3f}
    Seg Dice Score: {history['seg_dice'][-1]:.3f}
    
    Best Grade Acc: {max(history['grade_acc']):.3f}
    Best Seg Dice:  {max(history['seg_dice']):.3f}
    
    Final Losses:
      BBox:  {history['bbox_loss'][-1]:.4f}
      Class: {history['class_loss'][-1]:.4f}
      Seg:   {history['seg_loss'][-1]:.4f}
    """
    axes[1, 2].text(0.1, 0.5, final_stats, fontsize=12, 
                   verticalalignment='center', family='monospace',
                   bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.8))
    
    plt.tight_layout()
    plt.savefig('training_curves.png', dpi=300, bbox_inches='tight')
    print("✓ Training curves saved to training_curves.png")
    plt.close()


def analyze_multimodal_contribution(model, test_loader, device):
    """Analyze how multimodal features help each task"""
    print("\n" + "="*70)
    print("MULTIMODAL FEATURE CONTRIBUTION ANALYSIS")
    print("="*70)
    
    model.eval()
    
    # Helper function to compute metrics
    def compute_metrics(loader, use_clinical=True, use_imaging=True):
        grade_correct = 0
        total = 0
        dice_scores = []
        bbox_errors = []
        
        with torch.no_grad():
            for batch in loader:
                images = batch['image'].to(device)
                clinical = batch['clinical'].to(device)
                imaging = batch['imaging'].to(device)
                
                # Ablate features if needed
                if not use_clinical:
                    clinical = torch.zeros_like(clinical)
                if not use_imaging:
                    imaging = torch.zeros_like(imaging)
                
                bbox_true = batch['bbox'].to(device)
                mask_true = batch['mask'].to(device)
                has_bbox = batch['has_bbox'].to(device)
                grade = torch.tensor(batch['grade']).to(device)
                
                outputs = model(images, clinical, imaging)
                
                # Grade accuracy
                grade_pred = torch.argmax(outputs['grade'], dim=1)
                grade_correct += (grade_pred == grade).sum().item()
                total += len(grade)
                
                # Segmentation Dice
                seg_pred = torch.argmax(outputs['segmentation'], dim=1)
                for i in range(len(mask_true)):
                    if has_bbox[i] > 0:
                        pred = (seg_pred[i] == 1).float()
                        target = (mask_true[i] == 1).float()
                        intersection = (pred * target).sum()
                        dice = (2. * intersection + 1e-6) / (pred.sum() + target.sum() + 1e-6)
                        dice_scores.append(dice.item())
                
                # BBox error (IoU)
                bbox_pred = outputs['bbox']
                for i in range(len(bbox_true)):
                    if has_bbox[i] > 0:
                        # Compute IoU
                        pred = bbox_pred[i]
                        gt = bbox_true[i]
                        
                        # Get intersection
                        x_min = torch.max(pred[0], gt[0])
                        y_min = torch.max(pred[1], gt[1])
                        x_max = torch.min(pred[2], gt[2])
                        y_max = torch.min(pred[3], gt[3])
                        
                        inter_area = torch.clamp(x_max - x_min, min=0) * torch.clamp(y_max - y_min, min=0)
                        
                        pred_area = (pred[2] - pred[0]) * (pred[3] - pred[1])
                        gt_area = (gt[2] - gt[0]) * (gt[3] - gt[1])
                        union_area = pred_area + gt_area - inter_area
                        
                        iou = (inter_area / (union_area + 1e-6)).item()
                        bbox_errors.append(iou)
        
        grade_acc = grade_correct / total if total > 0 else 0
        avg_dice = np.mean(dice_scores) if dice_scores else 0
        avg_bbox_iou = np.mean(bbox_errors) if bbox_errors else 0
        
        return grade_acc, avg_dice, avg_bbox_iou
    
    # Baseline (all features)
    print("\n1. Full Multimodal (baseline):")
    grade_acc, seg_dice, bbox_iou = compute_metrics(test_loader, True, True)
    print(f"   Grade Accuracy: {grade_acc:.4f}")
    print(f"   Seg Dice:       {seg_dice:.4f}")
    print(f"   BBox IoU:       {bbox_iou:.4f}")
    
    # Without clinical
    print("\n2. Without Clinical Features:")
    grade_acc_nc, seg_dice_nc, bbox_iou_nc = compute_metrics(test_loader, False, True)
    print(f"   Grade Accuracy: {grade_acc_nc:.4f} (Δ = {grade_acc - grade_acc_nc:+.4f})")
    print(f"   Seg Dice:       {seg_dice_nc:.4f} (Δ = {seg_dice - seg_dice_nc:+.4f})")
    print(f"   BBox IoU:       {bbox_iou_nc:.4f} (Δ = {bbox_iou - bbox_iou_nc:+.4f})")
    
    # Without radiomic
    print("\n3. Without Radiomic Features:")
    grade_acc_ni, seg_dice_ni, bbox_iou_ni = compute_metrics(test_loader, True, False)
    print(f"   Grade Accuracy: {grade_acc_ni:.4f} (Δ = {grade_acc - grade_acc_ni:+.4f})")
    print(f"   Seg Dice:       {seg_dice_ni:.4f} (Δ = {seg_dice - seg_dice_ni:+.4f})")
    print(f"   BBox IoU:       {bbox_iou_ni:.4f} (Δ = {bbox_iou - bbox_iou_ni:+.4f})")
    
    # Image only
    print("\n4. Image Only (no tabular):")
    grade_acc_img, seg_dice_img, bbox_iou_img = compute_metrics(test_loader, False, False)
    print(f"   Grade Accuracy: {grade_acc_img:.4f} (Δ = {grade_acc - grade_acc_img:+.4f})")
    print(f"   Seg Dice:       {seg_dice_img:.4f} (Δ = {seg_dice - seg_dice_img:+.4f})")
    print(f"   BBox IoU:       {bbox_iou_img:.4f} (Δ = {bbox_iou - bbox_iou_img:+.4f})")
    
    print("\n" + "="*70)
    print("INTERPRETATION")
    print("="*70)
    
    multimodal_gain_grade = grade_acc - grade_acc_img
    multimodal_gain_seg = seg_dice - seg_dice_img
    multimodal_gain_bbox = bbox_iou - bbox_iou_img
    
    print(f"\nMultimodal Gain (vs Image-only):")
    print(f"  Classification: {multimodal_gain_grade:+.4f}")
    print(f"  Segmentation:   {multimodal_gain_seg:+.4f}")
    print(f"  Detection:      {multimodal_gain_bbox:+.4f}")
    
    if multimodal_gain_grade > 0.05 or multimodal_gain_seg > 0.05:
        print("\n✅ SUCCESS! Multimodal features significantly improve performance!")
    elif multimodal_gain_grade > 0.02 or multimodal_gain_seg > 0.02:
        print("\n✓ GOOD: Multimodal features provide moderate improvement.")
    else:
        print("\n⚠️ NEUTRAL: Limited multimodal benefit (small dataset).")
    
    print("\n" + "="*70)


# ==================== MAIN ====================

def main():
    print("\n" + "="*70)
    print(" "*15 + "DUKE MULTI-TASK MULTIMODAL MODEL")
    print("="*70)
    print("\n🎯 Combined Tasks:")
    print("   1. Tumor Detection (bounding box)")
    print("   2. Tumor Classification (grade, subtype)")
    print("   3. Tumor Segmentation (pixel-wise mask)")
    print("\n📊 Multimodal Features:")
    print("   • MRI images")
    print("   • Clinical features (staging, receptors)")
    print("   • Radiomic features (size, texture)")
    print("\n💡 This demonstrates the FULL power of multimodal learning!\n")
    
    # Configuration
    EXPORT_DIR = 'data'
    BATCH_SIZE = 4
    NUM_EPOCHS = 50
    LEARNING_RATE = 1e-4
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    print(f"Device: {DEVICE}")
    print(f"Batch Size: {BATCH_SIZE}")
    print(f"Learning Rate: {LEARNING_RATE}\n")
    
    # Transforms
    transform_train = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(10),
        transforms.ColorJitter(brightness=0.2, contrast=0.2),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                           std=[0.229, 0.224, 0.225])
    ])
    
    transform_val = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                           std=[0.229, 0.224, 0.225])
    ])
    
    # Load datasets
    print("="*70)
    print("LOADING DATASETS")
    print("="*70)
    
    train_dataset = DukeMultiTaskDataset(
        Path(EXPORT_DIR) / 'train',
        transform=transform_train,
        use_clinical=True,
        use_imaging_features=True
    )
    
    test_dataset = DukeMultiTaskDataset(
        Path(EXPORT_DIR) / 'test',
        transform=transform_val,
        use_clinical=True,
        use_imaging_features=True
    )
    
    # Share statistics
    test_dataset.clinical_mean = train_dataset.clinical_mean
    test_dataset.clinical_std = train_dataset.clinical_std
    test_dataset.imaging_mean = train_dataset.imaging_mean
    test_dataset.imaging_std = train_dataset.imaging_std
    
    print(f"\nTrain: {len(train_dataset)} | Test: {len(test_dataset)}")
    
    if len(train_dataset) == 0 or len(test_dataset) == 0:
        print("\n❌ ERROR: No samples loaded!")
        return
    
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    
    # Model
    print("\n" + "="*70)
    print("INITIALIZING MULTI-TASK MODEL")
    print("="*70)
    
    model = MultiTaskMultimodalModel(clinical_dim=10, imaging_dim=12).to(DEVICE)
    
    total_params = sum(p.numel() for p in model.parameters())
    print(f"Model: MultiTaskMultimodalModel")
    print(f"Total parameters: {total_params:,}")
    print(f"Architecture: ResNet50 backbone + Multi-task heads")
    
    # Train
    trainer = MultiTaskTrainer(model, DEVICE, lr=LEARNING_RATE)
    trainer.train(train_loader, test_loader, num_epochs=NUM_EPOCHS, patience=10)
    
    # Visualizations
    print("\n" + "="*70)
    print("GENERATING VISUALIZATIONS")
    print("="*70)
    
    plot_training_curves(trainer.history)
    visualize_multitask_results(model, test_loader, DEVICE, num_samples=4)
    
    # Feature importance
    analyze_multimodal_contribution(model, test_loader, DEVICE)
    
    # Summary
    print("\n" + "="*70)
    print("FINAL SUMMARY")
    print("="*70)
    
    print("\n✓ Training complete!")
    print("✓ Model saved: best_multitask_model.pth")
    print("✓ Training curves: training_curves.png")
    print("✓ Multi-task results: multitask_results.png")
    
    print(f"\n📊 Final Performance:")
    print(f"   Classification (Grade): {trainer.history['grade_acc'][-1]:.1%}")
    print(f"   Segmentation (Dice):    {trainer.history['seg_dice'][-1]:.1%}")
    print(f"   Detection (BBox Loss):  {trainer.history['bbox_loss'][-1]:.4f}")
    
    print("\n💡 This model simultaneously:")
    print("   • Locates tumors (detection)")
    print("   • Characterizes them (classification)")
    print("   • Delineates them (segmentation)")
    print("\n🎉 All using multimodal MRI + clinical + radiomic data!")
    
    print("\n" + "="*70)


if __name__ == '__main__':
    main()


               DUKE MULTI-TASK MULTIMODAL MODEL

🎯 Combined Tasks:
   1. Tumor Detection (bounding box)
   2. Tumor Classification (grade, subtype)
   3. Tumor Segmentation (pixel-wise mask)

📊 Multimodal Features:
   • MRI images
   • Clinical features (staging, receptors)
   • Radiomic features (size, texture)

💡 This demonstrates the FULL power of multimodal learning!

Device: cuda
Batch Size: 4
Learning Rate: 0.0001

LOADING DATASETS

Dataset Statistics:
  Total samples: 20
  Samples with bounding boxes: 20

Dataset Statistics:
  Total samples: 5
  Samples with bounding boxes: 5

Train: 20 | Test: 5

INITIALIZING MULTI-TASK MODEL
Model: MultiTaskMultimodalModel
Total parameters: 52,378,766
Architecture: ResNet50 backbone + Multi-task heads

MULTI-TASK TRAINING
Tasks: Detection + Classification + Segmentation

Epoch 1/50
  Train Loss: 0.6347 | Val Loss: 0.6442
  BBox Loss:  0.0153
  Class Loss: 0.9406
  Seg Loss:   0.7423
  Grade Acc:  0.6000
  Seg Dice:   0.0375
  ✓ Best model sav